In [ ]:
import torch
import torch.nn as nn
import numpy as np
import time
from scipy.stats import norm
import matplotlib.pyplot as plt
from functools import reduce
import math

# -------------------------
# Hyperparameters & Market
# -------------------------
time_final = 1.0
S_max = 160.0
r = 0.05
sigma = 0.2
K = 40.0

# QPINN settings
num_qubits = 2
num_layers_q = 4
learning_rate = 1e-2
num_iterations = 3000
n_interior = 128
n_boundary = 128
observable_scale = 50

# Choose device
device = torch.device("cpu" if torch.backends.mps.is_available() else "cpu")

# -------------------------
# Pure-Torch QNode Replacement (real-only simulation)
# -------------------------
# We'll represent state as real and imaginary parts separately, all float32

# Prebuilt identity and Pauli X,Z as real matrices
dtype = torch.float32
I2 = torch.eye(2, dtype=dtype, device=device)
X = torch.tensor([[0,1],[1,0]], dtype=dtype, device=device)
Z = torch.tensor([[1,0],[0,-1]], dtype=dtype, device=device)
# RY uses Pauli Y; real representation of Y = [[0,-i],[i,0]] so real=0, imag = [[0,-1],[1,0]]
y_im = torch.tensor([[0,-1],[1,0]], dtype=dtype, device=device)

# Two-qubit CNOT: purely real
CNOT = torch.tensor([
    [1,0,0,0], [0,1,0,0], [0,0,0,1], [0,0,1,0]
], dtype=dtype, device=device)

# Helper to kron two real-imag blocks (A,B)⊗(C,D)
def kron2(A_real, A_imag, B_real, B_imag):
    # (A + i B)⊗(C + i D) = (A⊗C - B⊗D) + i(A⊗D + B⊗C)
    real = torch.kron(A_real, B_real) - torch.kron(A_imag, B_imag)
    imag = torch.kron(A_real, B_imag) + torch.kron(A_imag, B_real)
    return real, imag

# Rotation gates produce real and imag parts

def rx_realimag(theta):
    # RX = cos(t/2) I - i sin(t/2) X
    c = torch.cos(theta/2)
    s = torch.sin(theta/2)
    A = c * I2
    B = s * X   # because -i*X => imag = +X
    return A, B

def rz_realimag(theta):
    # RZ = cos(t/2) I - i sin(t/2) Z
    c = torch.cos(theta/2)
    s = torch.sin(theta/2)
    A = c * I2
    B = s * Z
    return A, B

def ry_realimag(theta):
    # RY = cos(t/2) I - i sin(t/2) Y
    c = torch.cos(theta/2)
    s = torch.sin(theta/2)
    A = c * I2
    B = s * y_im
    return A, B

# Observable: averaged projector P0 on each qubit (real)
def Observable_local_real(n):
    P0 = torch.tensor([[1.,0],[0,0]], dtype=dtype, device=device)
    OL = torch.zeros((2**n, 2**n), dtype=dtype, device=device)
    for j in range(n):
        # build P0 on j
        factor = [I2]*n
        factor[j] = P0
        mat = reduce(lambda a,b: torch.kron(a,b), factor)
        OL += mat
    return OL/n

# Feature map as real and imag 4x4

def feature_map_realimag(s, t):
    th_s = s * math.pi / S_max
    th_t = t * math.pi / time_final
    A1,B1 = rx_realimag(th_s)
    A2,B2 = rx_realimag(th_t)
    return kron2(A1,B1, A2,B2)

# Variational block
def variational_realimag(w):
    # w: (2,3) for 2 qubits; real-imag unitaries
    zero2 = torch.zeros_like(I2)
    # start as identity
    Rr = torch.eye(4, dtype=dtype, device=device)
    Ri = torch.zeros((4,4), dtype=dtype, device=device)
    # apply rotations on qubit 0: RZ, RY, RZ
    for idx, rot_fn in enumerate([rz_realimag, ry_realimag, rz_realimag]):
        A, B = rot_fn(w[0, idx])
        Ar, Ai = kron2(A, B, I2, zero2)
        Rr, Ri = Ar @ Rr - Ai @ Ri, Ar @ Ri + Ai @ Rr
    # apply rotations on qubit 1: RZ, RY, RZ
    for idx, rot_fn in enumerate([rz_realimag, ry_realimag, rz_realimag]):
        A, B = rot_fn(w[1, idx])
        Ar, Ai = kron2(I2, zero2, A, B)
        Rr, Ri = Ar @ Rr - Ai @ Ri, Ar @ Ri + Ai @ Rr
    # entangle with CNOT (real only)
    Rr = CNOT @ Rr
    Ri = CNOT @ Ri
    return Rr, Ri

class QPINN(nn.Module):
    def __init__(self, num_layers, num_qubits):
        super().__init__()
        self.num_layers = num_layers
        self.nq = num_qubits
        self.weights = nn.Parameter(0.1*torch.randn((num_layers, num_qubits,3), device=device))
        # init |00> real
        self.register_buffer('_s0', torch.tensor([1.,0,0,0], dtype=dtype, device=device))
        self.register_buffer('_s1', torch.zeros(4, dtype=dtype, device=device))
        obs = Observable_local_real(num_qubits)*observable_scale
        self.register_buffer('_obs', obs)

    def forward(self, X):
        out = []
        for s,t in X:
            sr, si = self._s0, self._s1  # real, imag state
            # feature
            Ar, Ai = feature_map_realimag(s,t)
            # apply layer
            for i in range(self.num_layers):
                nr = Ar@sr - Ai@si
                ni = Ar@si + Ai@sr
                # variational block
                Vr, Vi = variational_realimag(self.weights[i])
                tr = Vr@nr - Vi@ni
                ti = Vr@ni + Vi@nr
                sr, si = tr, ti
                # reapply feature
                nr = Ar@sr - Ai@si
                ni = Ar@si + Ai@sr
                sr, si = nr, ni
            # expectation: sr^T O sr + si^T O si
            val = sr @ (self._obs @ sr) + si @ (self._obs @ si)
            out.append(val.unsqueeze(-1))
        return torch.stack(out)

# The rest (PINN, training loop, evaluation) remains identical, now all float32


# -------------------------
# Classical PINN definition
# -------------------------
class PINN(nn.Module):
    def __init__(self, hidden=100, layers=8):
        super().__init__()
        layers_list = [nn.Linear(2, hidden), nn.LeakyReLU(0.1)]
        for _ in range(layers):
            layers_list += [nn.Linear(hidden, hidden), nn.Tanh()]
        layers_list += [nn.Linear(hidden, 1)]
        self.net = nn.Sequential(*layers_list)
    def forward(self, x):
        return self.net(x)

# Analytical BS formula (works on torch tensors)
def bs_call(S, t):
    tau = time_final - t
    d1 = (torch.log(S/K) + (r + 0.5*sigma**2)*tau) / (sigma*torch.sqrt(tau))
    d2 = d1 - sigma*torch.sqrt(tau)
    C = torch.where(
        tau>0,
        S*0 + S*torch.from_numpy(norm.cdf(d1.cpu().numpy())).to(device)
        - K*torch.exp(-r*tau)*torch.from_numpy(norm.cdf(d2.cpu().numpy())).to(device),
        torch.maximum(S-K, torch.zeros_like(S))
    )
    return C

# Move models to device
q_model = QPINN(num_layers_q, num_qubits).to(device)
c_model = PINN(hidden=10, layers=2).to(device)

o_q = torch.optim.Adam(q_model.parameters(), lr=learning_rate)
o_c = torch.optim.Adam(c_model.parameters(), lr=learning_rate)

def mse(a, b): return torch.mean((a-b)**2)

# Loss including PDE and BCs/terminal, all tensors on `device`
from torch.autograd import grad

def compute_losses(model):
    # interior
    S_int = torch.rand(n_interior,1, device=device)*S_max
    t_int = torch.rand(n_interior,1, device=device)*time_final
    S_int.requires_grad_(True)
    t_int.requires_grad_(True)
    X_int = torch.cat([S_int, t_int], dim=1)

    V_int = model(X_int)
    V_t = grad(V_int, t_int, grad_outputs=torch.ones_like(V_int), create_graph=True)[0]
    V_S = grad(V_int, S_int, grad_outputs=torch.ones_like(V_int), create_graph=True)[0]
    V_SS = grad(V_S, S_int, grad_outputs=torch.ones_like(V_S), create_graph=True)[0]

    residual = V_t + 0.5*sigma**2*S_int**2*V_SS + r*S_int*V_S - r*V_int
    loss_pde = torch.mean(residual**2)

    # terminal
    S_term = torch.linspace(0, S_max, n_boundary, device=device).unsqueeze(1)
    t_term = torch.full_like(S_term, time_final)
    V_term = model(torch.cat([S_term, t_term],1))
    V_ex_t = torch.maximum(S_term-K, torch.zeros_like(S_term))
    loss_term = mse(V_term, V_ex_t)

    # S=0
    t_b0 = torch.rand(n_boundary,1, device=device)*time_final
    V_b0 = model(torch.cat([torch.zeros_like(t_b0), t_b0],1))
    loss_b0 = mse(V_b0, torch.zeros_like(V_b0))

    # S=S_max
    t_b1 = torch.rand(n_boundary,1, device=device)*time_final
    V_b1 = model(torch.cat([torch.full_like(t_b1, S_max), t_b1],1))
    V_ex_b1 = S_max - K*torch.exp(-r*(time_final-t_b1))
    loss_b1 = mse(V_b1, V_ex_b1)

    return loss_pde + loss_term + loss_b0 + loss_b1

# -------------------------
# Training Loop
# -------------------------
loss_hist_q, loss_hist_c = [], []
print("Training on device:", device)
for it in range(num_iterations+1):
    # QPINN update
    start = time.perf_counter()
    o_q.zero_grad()
    lq = compute_losses(q_model)
    lq.backward()
    o_q.step()
    end   = time.perf_counter()
    print(f"Execution time: {end - start:.6f} seconds")

    # PINN update
    o_c.zero_grad()
    lc = compute_losses(c_model)
    lc.backward()
    o_c.step()

    loss_hist_q.append(lq.item())
    loss_hist_c.append(lc.item())
    if it % 100 == 0:
        print(f"Iter {it}: QPINN={lq:.3e}, PINN={lc:.3e}")
print("Done training.")

# -------------------------
# Evaluation & Plotting
# -------------------------
S_vals = torch.linspace(0, S_max, 50, device=device)
T_vals = torch.linspace(0, time_final, 50, device=device)
Sg, Tg = torch.meshgrid(S_vals, T_vals, indexing='ij')
X_eval = torch.cat([Sg.reshape(-1,1), Tg.reshape(-1,1)],1)
with torch.no_grad():
    Vq = q_model(X_eval).reshape(50,50).cpu().numpy()
    Vc = c_model(X_eval).reshape(50,50).cpu().numpy()
    Van = bs_call(X_eval[:,0], X_eval[:,1]).reshape(50,50).cpu().numpy()

fig = plt.figure(figsize=(15,5))
for idx, (Z, title) in enumerate([(Van, 'Analytical'), (Vq, 'QPINN'), (Vc, 'PINN')]):
    ax = fig.add_subplot(1,3,idx+1, projection='3d')
    ax.plot_surface(Tg.cpu().numpy(), Sg.cpu().numpy(), Z, edgecolor='none')
    ax.set_title(title)
    ax.set_xlabel('t')
    ax.set_ylabel('S')
    ax.set_zlabel('V')
plt.tight_layout()
plt.show()


In [ ]:
trainable_params = sum(p.numel() for p in q_model.parameters() if p.requires_grad)
print(f"QPINN trainable parameters: {trainable_params}")
trainable_params = sum(p.numel() for p in c_model.parameters() if p.requires_grad)
print(f"PINN trainable parameters: {trainable_params}")